In [2]:
import os
os.environ["KERAS_BACKEND"]="tensorflow"
import keras
os.environ['GCS_READ_BUCKET_ANONYMOUSLY'] = '1'
import tensorflow as tf
filename= keras.utils.get_file(
    origin=(
        "https://storage.googleapis.com/download.tensorflow.org/"
        "data/shakespeare.txt"
    ),
)
shakespear= open(filename,"r").read()   

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
shakespear[:250]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n'

In [5]:
sequence_length=100

def split_input(input, sequence_length):
    for i in range(0, len(input),sequence_length):
        yield input[i:i+sequence_length]
features= list(split_input(shakespear[:-1],sequence_length))
labels= list(split_input(shakespear[1:],sequence_length))
dataset= tf.data.Dataset.from_tensor_slices((features,labels))

In [6]:
x,y = next(dataset.as_numpy_iterator())
x[:50], y[:50]

(b'First Citizen:\nBefore we proceed any further, hear',
 b'irst Citizen:\nBefore we proceed any further, hear ')

In [8]:
from keras import layers

tokenizer= layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text,labels: text))
    

2026-06-17 23:00:14.285661: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [12]:
vocabulary_size=tokenizer.vocabulary_size()

In [10]:
dataset= dataset.map(
    lambda features,labels: (tokenizer(features),tokenizer(labels)),
    num_parallel_calls=8,
)
training_data= dataset.shuffle(10_000).batch(64).cache()

In [13]:
embedding_dim=256
hidden_dim=1024

inputs= keras.Input(shape=(sequence_length,),dtype="int",name="token_ids")
x=layers.Embedding(vocabulary_size,embedding_dim)(inputs)
x= layers.GRU(hidden_dim,return_sequences=True)(x)
x=layers.Dropout(0.1)(x)
outputs= layers.Dense(vocabulary_size, activation="softmax")(x)
model= keras.Model(inputs,outputs)

In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)          │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 256)       │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 100, 1024)      │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 1024)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100, 67)        │        68,675 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data,epochs=20)

Epoch 1/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 76s 429ms/step - loss: 2.5522 - sparse_categorical_accuracy: 0.3129
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 88s 505ms/step - loss: 1.8693 - sparse_categorical_accuracy: 0.4490
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 97s 553ms/step - loss: 1.6320 - sparse_categorical_accuracy: 0.5127
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 97s 557ms/step - loss: 1.5060 - sparse_categorical_accuracy: 0.5458
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 101s 577ms/step - loss: 1.4284 - sparse_categorical_accuracy: 0.5654
Epoch 6/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 109s 620ms/step - loss: 1.3724 - sparse_categorical_accuracy: 0.5794
Epoch 7/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 139s 796ms/step - loss: 1.3261 - sparse_categorical_accuracy: 0.5917
Epoch 8/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 153s 872ms/step - loss: 1.2861 - sparse_categorical_accuracy: 0.6015
Epoch 9/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 142s 810ms/step - loss: 1.2478 - sparse_categorical_accuracy: 0.6119
Epoch 10/20
175/175 ━━━

# Autoregressive Model

In [17]:
inputs= keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state= keras.Input(shape=(hidden_dim,), name="state")
x= layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state= layers.GRU(hidden_dim, return_state=True)(
    x, initial_state= input_state
)
outputs= layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model= keras.Model(
    inputs=(inputs,input_state),
    outputs=(outputs, output_state),
)
generation_model.set_weights(model.get_weights())

In [30]:
tokens= tokenizer.get_vocabulary()
token_ids= range(vocabulary_size)
char_to_id= dict(zip(tokens,token_ids))
id_to_char= dict(zip(token_ids,tokens))
prompt="""
Nyaa  dont you just love mochi ice cream Konkeo chan?
Oh yeah, mochi ice creams very good, Raku-chan.
Desu Ah, Im really full now. Come on, lets go home,
"""

In [31]:
input_ids= [char_to_id[c] for c in prompt]
state= keras.ops.zeros(shape=(1,hidden_dim))
for token_ids in input_ids:
    inputs= keras.ops.expand_dims([token_ids],axis=0)
    predictions, state= generation_model.predict((inputs,state),verbose=0)


In [33]:
import numpy as np

generated_ids=[]
max_length= 400
for i in range(max_length):
    next_char= int(np.argmax(predictions,axis=-1)[0])
    generated_ids.append(next_char)
    inputs= keras.ops.expand_dims([next_char],axis=0)
    predictions,state= generation_model.predict((inputs,state),verbose=0)
output= "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)


Nyaa  dont you just love mochi ice cream Konkeo chan?
Oh yeah, mochi ice creams very good, Raku-chan.
Desu Ah, Im really full now. Come on, lets go home,
counterfect
That we came from Rutland home again.

BUCKINGHAM:
Whener, a gentlewoman, our cousin, cousin Hereford?

First Citizen:
We cannot be a prince of traitors for the time of death,
And then my son's an ears to my fortune and the state
To chate and speak to the sea for to the gentle on
An once or endures in the midst of this.

FRIAR LAURENCE:
So come to make me with a living foot,
And then t
